<a href="https://colab.research.google.com/github/smeason/pushT-project/blob/main/CS5180_PushT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [117]:
# !pip install gym-pusht

In [118]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.lines import Line2D
import numpy as np
from pathlib import Path
import time

In [119]:
print(matplotlib.get_backend(), flush=True)
fig, ax = plt.subplots()
print("subplots succeeded", flush=True)
plt.close(fig)

Agg
subplots succeeded


In [120]:
# example.py
import gymnasium as gym
import gym_pusht

# env = gym.make("gym_pusht/PushT-v0",
#                render_mode='rgb_array',
#                obs_type="state")
# observation, info = env.reset()

# for _ in range(10):
#     action = env.action_space.sample()
#     observation, reward, terminated, truncated, info = env.step(action)
#     image = env.render()

#     if terminated or truncated:
#         observation, info = env.reset()

# env.close()

In [121]:
from gymnasium import ActionWrapper, spaces

### Wrap pushT environment to use a velocity-based action space (instead of position)
class PushTVelocityWrapper(ActionWrapper):
    def __init__(self, env, max_speed=40.0, min_speed=5.0):
        super().__init__(env)

        self.max_speed = max_speed
        self.min_speed = min_speed
        self.action_space = spaces.Box(
            low=-1.0,
            high=1.0,
            shape=env.action_space.shape,
            dtype=np.float32,
        )

    def action(self, normalized_velocity):
        current_position = np.asarray(self.unwrapped.agent.position, dtype=np.float32)

        velocity = np.clip(normalized_velocity, -1.0, 1.0) * self.max_speed
        speed = np.linalg.norm(velocity)

        # Floor nonzero commands; leave a zero command as a stop.
        if 1e-6 < speed < self.min_speed:
            velocity *= self.min_speed / speed
            
        target_position = current_position + velocity / self.unwrapped.control_hz

        low = self.env.action_space.low
        high = self.env.action_space.high
        return np.clip(target_position, low, high).astype(np.float32)

In [122]:
class DistanceCurriculumAndRewardWrapper(gym.Wrapper):
    def __init__(
        self,
        env,#: gym.Env,
        max_distance=30,
        max_angle=np.deg2rad(15),
        max_attempts=20_000,
        agent_object_weight=0.1,
        object_goal_weight=1.0,
        object_goal_angle_weight=1.0,
        contact_bonus=0.1,
        distance_scale=512.0,
    ):
        super().__init__(env)
        self.max_attempts = max_attempts
        self.agent_object_weight = agent_object_weight
        self.object_goal_weight = object_goal_weight
        self.object_goal_angle_weight = object_goal_angle_weight
        self.contact_bonus = contact_bonus
        self.distance_scale = distance_scale
        self.set_limits(max_distance, max_angle)

    def _metrics(self, info):
        agent_xy = np.asarray(info["pos_agent"][:2])
        object_xy = np.asarray(info["block_pose"][:2])
        goal_xy = np.asarray(info["goal_pose"][:2])
        angle_delta = info["block_pose"][2] - info["goal_pose"][2]
        angle_difference = float(np.arctan2(np.sin(angle_delta), np.cos(angle_delta)))

        # Check the pusher's shape against the block's shapes.
        base = self.unwrapped
        contact = any(
            hit.shape.body is base.block
            for agent_shape in base.agent.shapes
            for hit in base.space.shape_query(agent_shape)
        )

        return {
            "agent_object_distance": float(np.linalg.norm(agent_xy - object_xy)),
            "object_goal_distance": float(np.linalg.norm(object_xy - goal_xy)),
            "object_goal_angle_difference": angle_difference,
            "object_goal_angle_error": abs(angle_difference),
            "contact": bool(contact),
        }

    def set_limits(self, max_distance, max_angle):
        """Distance is in environment units; angle is in radians."""
        if max_distance < 0:
            raise ValueError("max_distance must be nonnegative")
        if not 0 <= max_angle <= np.pi:
            raise ValueError("max_angle must be between 0 and pi")

        self.max_distance = max_distance
        self.max_angle = max_angle

    def reset(self, *, seed=None, options=None):
        for attempt in range(self.max_attempts):
            obs, info = self.env.reset(
                seed=seed if attempt == 0 else None,
                options=options,
            )

            base = self.env.unwrapped

            block_xy = np.asarray(base.block.position)
            goal_xy = np.asarray(base.goal_pose[:2])
            distance = np.linalg.norm(block_xy - goal_xy)

            block_angle = base.block.angle
            goal_angle = base.goal_pose[2]
            delta = block_angle - goal_angle
            angle_error = abs(np.arctan2(np.sin(delta), np.cos(delta)))

            if distance <= self.max_distance and angle_error <= self.max_angle:
                info = dict(info)
                info["initial_goal_distance"] = float(distance)
                info["initial_angle_error"] = float(angle_error)
                metrics = self._metrics(info)
                self._previous = metrics
                self._was_contact = metrics["contact"]
                # print(f'attempt num: {attempt}')
                return obs, {**info, **metrics}

        raise RuntimeError(
            "Could not sample a reset satisfying both limits. "
            "Try relaxing them or increasing max_attempts."
        )

    def step(self, action):
        observation, base_reward, terminated, truncated, info = self.env.step(action)
        metrics = self._metrics(info)

        base_reward = float(base_reward) * 0.25

        # to incentivize contact initially in learning, reward based off distance to object
        dist_reward = (1.0 - metrics["agent_object_distance"] / self.distance_scale) * 0.2

        # Reward progress (distance reductions), scaled to the 512x512 arena.
        progress_reward = 40*(
            # self.agent_object_weight
            # * (self._previous["agent_object_distance"] - metrics["agent_object_distance"])
            self.object_goal_weight
            * (self._previous["object_goal_distance"] - metrics["object_goal_distance"])
        ) / self.distance_scale
        progress_reward = max(-1.0, min(progress_reward, 1.0)) # clip between -1 and 1

        angle_progress_reward = 20 * self.object_goal_angle_weight * (
            self._previous["object_goal_angle_error"] - metrics["object_goal_angle_error"]
        ) / np.pi
        angle_progress_reward = max(-1.0, min(angle_progress_reward, 1.0)) # clip between -1 and 1

        # Give a one-time bonus when contact is first made.
        contact_reward = self.contact_bonus * float(
            metrics["contact"] and not self._was_contact
        )

        reward = float(base_reward + dist_reward + progress_reward + angle_progress_reward + contact_reward) - 0.1
        info = {
            **info,
            **metrics,
            "dist_reward": float(dist_reward),
            "base_reward": float(base_reward),
            "progress_reward": float(progress_reward),
            "angle_progress_reward": float(angle_progress_reward),
            "contact_reward": float(contact_reward),
        }

        self._previous = metrics
        self._was_contact = metrics["contact"]
        return observation, reward, terminated, truncated, info


In [ ]:
"""Parameters"""
policy_type: str = "mlp"
state_dim: int = 5
action_dim: int = 2
lr_actor: float = 0.0005
lr_critic: float = 0.0005
max_ppo_steps: int = 100
max_episode_len: int = 500
clip: float = 0.2
discount_gamma: float = 0.99
gae_lambda: float = 0.95
update_epochs: int = 5
batch_size: int = 20
entropy_coefficient: float = 0.01

initial_max_dist: float = 40
initial_max_angle = 30 # degrees
max_speed = 60 # maximum speed of pusher arm
min_speed = 20
action_every_n_steps = 4

In [124]:
env = DistanceCurriculumAndRewardWrapper(
    gym.make("gym_pusht/PushT-v0",
             render_mode="rgb_array",
             obs_type="state"),
    max_distance=initial_max_dist,
    max_angle=np.deg2rad(initial_max_angle),
)

env = PushTVelocityWrapper(env, max_speed=max_speed, min_speed=min_speed)

print(f'{env.observation_space} | shape: {env.observation_space.shape[0]}')
print(f'{env.action_space} | shape: {env.action_space.shape[0]}')

obs, info = env.reset()
print("Distance:", info["initial_goal_distance"])
print("Angle error (degrees):", np.rad2deg(info["initial_angle_error"]))

# Increase difficulty between episodes:
# env.set_limits(max_distance=60, max_angle=np.deg2rad(30))

Box(0.0, [512.         512.         512.         512.           6.28318531], (5,), float64) | shape: 5
Box(-1.0, 1.0, (2,), float32) | shape: 2
Distance: 23.74267420357637
Angle error (degrees): 16.93354653654613


In [125]:
# @title
############# print all hyperparameters #############

print("--------------------------------------------------------------------------------------------")

print("max training timesteps : ", max_ppo_steps)
print("max timesteps per episode : ", max_episode_len)

# print("model saving frequency : " + str(save_model_freq) + " timesteps")
# print("log frequency : " + str(log_freq) + " timesteps")
# print("printing average reward over episodes in last : " + str(print_freq) + " timesteps")

print("--------------------------------------------------------------------------------------------")

print("state space dimension : ", state_dim)
print("action space dimension : ", action_dim)

print("--------------------------------------------------------------------------------------------")

print("Initializing a continuous action space policy")
print("--------------------------------------------------------------------------------------------")
# print("starting std of action distribution : ", action_std)
# print("decay rate of std of action distribution : ", action_std_decay_rate)
# print("minimum std of action distribution : ", min_action_std)
# print("decay frequency of std of action distribution : " + str(action_std_decay_freq) + " timesteps")

print("--------------------------------------------------------------------------------------------")

# print("PPO update frequency : " + str() + " timesteps")
print("PPO K epochs : ", update_epochs)
print("PPO epsilon clip : ", clip)
print("discount factor (gamma) : ", discount_gamma)

print("--------------------------------------------------------------------------------------------")

print("optimizer learning rate actor : ", lr_actor)
print("optimizer learning rate critic : ", lr_critic)

--------------------------------------------------------------------------------------------
max training timesteps :  100
max timesteps per episode :  500
--------------------------------------------------------------------------------------------
state space dimension :  5
action space dimension :  2
--------------------------------------------------------------------------------------------
Initializing a continuous action space policy
--------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------
PPO K epochs :  5
PPO epsilon clip :  0.2
discount factor (gamma) :  0.99
--------------------------------------------------------------------------------------------
optimizer learning rate actor :  0.0005
optimizer learning rate critic :  0.0005


In [126]:
import copy

import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal


class RolloutBuffer:
    def __init__(self):
        self.actions = []
        self.states = []
        self.logprobs = []
        self.rewards = []
        self.state_values = []
        self.next_states = []
        self.durations = []
        self.terminateds = []
        self.truncateds = []

    def clear(self):
        self.actions.clear()
        self.states.clear()
        self.logprobs.clear()
        self.rewards.clear()
        self.state_values.clear()
        self.next_states.clear()
        self.durations.clear()
        self.terminateds.clear()
        self.truncateds.clear()


class ActorCritic(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, action_std: float = 0.5):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, action_dim),
            nn.Tanh(),
        )
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        self.log_std = nn.Parameter(torch.full((action_dim,), np.log(action_std)))

    def get_distribution(self, state: torch.Tensor) -> Normal:
        mean = self.actor(state)
        std = self.log_std.exp().expand_as(mean)
        return Normal(mean, std)

    def act(self, state: torch.Tensor):
        dist = self.get_distribution(state)
        action = dist.sample()
        return action, dist.log_prob(action).sum(dim=-1), self.critic(state).squeeze(-1)

    def evaluate(self, state: torch.Tensor, action: torch.Tensor):
        dist = self.get_distribution(state)
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        value = self.critic(state).squeeze(-1)
        return log_prob, value, entropy


class PPO:
    def __init__(
        self,
        env,
        policy_type: str = "mlp",
        state_dim: int = 5,
        action_dim: int = 2,
        lr_actor: float = 0.0003,
        lr_critic: float = 0.0003,
        max_ppo_steps: int = 100,
        max_episode_len: int = 200,
        clip: float = 0.2,
        discount_gamma: float = 0.99,
        gae_lambda: float = 0.90,
        update_epochs: int = 10,
        batch_size: int = 20,
        entropy_coefficient: float = 0.01,
        action_every_n_steps: int = 4
    ) -> None:
        self.env = env
        self.policy_type = policy_type
        if policy_type != "mlp":
            raise NotImplementedError("Only the MLP policy is implemented")

        self.state_dim = state_dim if state_dim is not None else env.observation_space.shape[0]
        self.action_dim = action_dim if action_dim is not None else env.action_space.shape[0]
        self.max_steps = max_ppo_steps
        self.MAX_EPISODE_LEN = max_episode_len
        self.clip = clip
        self.discount_gamma = discount_gamma
        self.gae_lambda = gae_lambda
        self.update_epochs = update_epochs
        self.num_rollouts = batch_size
        self.entropy_coefficient = entropy_coefficient
        if action_every_n_steps < 1:
            raise ValueError("action_every_n_steps must be at least 1")
        self.action_every_n_steps = action_every_n_steps
        self.buffer = RolloutBuffer()

        self.policy = ActorCritic(self.state_dim, self.action_dim)
        self.policy_old = copy.deepcopy(self.policy)
        self.policy_old.eval()

        self.optimizer = torch.optim.Adam([
            {"params": self.policy.actor.parameters(), "lr": lr_actor},
            {"params": self.policy.critic.parameters(), "lr": lr_critic},
            {"params": self.policy.log_std, "lr": lr_actor},
        ])
        self.loss_fn = nn.MSELoss()

    def _to_environment_action(self, action: torch.Tensor) -> np.ndarray:
        # convert raw action tensor output to usable action numpy array
        action = action.detach().cpu().numpy()
        if action.ndim == 2:
            action = action[0]

        low = np.asarray(self.env.action_space.low, dtype=np.float32)
        high = np.asarray(self.env.action_space.high, dtype=np.float32)
        scaled = low + (high - low) * (np.clip(action, -1.0, 1.0) + 1.0) / 2.0
        return scaled.astype(np.float32)

    def collect_dataset(self):
        self.buffer.clear()
        self.policy_old.eval()

        start_pose = None
        end_pose = None
        trajectory = []
        total_rewards = []
        episode_lengths = []

        for i in range(self.num_rollouts):
            observation, _ = self.env.reset()
            if i == 0:
                start_pose = self.env.render()
                # start_pose = observation[2:]

            ep_reward = 0
            ep_len = 0

            while ep_len < self.MAX_EPISODE_LEN:
                state = torch.as_tensor(observation, dtype=torch.float32).unsqueeze(0)
                with torch.no_grad():
                    action, log_prob, state_value = self.policy_old.act(state)

                environment_action = self._to_environment_action(action)
                macro_reward = 0.0
                duration = 0
                steps_to_run = min(self.action_every_n_steps, self.MAX_EPISODE_LEN - ep_len)

                terminated = False
                truncated = False
                for j in range(steps_to_run):
                    next_observation, reward, terminated, truncated, info = self.env.step(environment_action)
                    if i == 0 and j == 0 and ep_len % 100 == 0:
                        print(f'dist_reward {info["dist_reward"]} | base_reward: {info["base_reward"]} | progress_reward: {info["progress_reward"]} | angle_progress_reward: {info["angle_progress_reward"]}')
                    done = terminated or truncated

                    macro_reward += (self.discount_gamma ** j) * float(reward)
                    ep_reward += float(reward)
                    ep_len += 1
                    duration += 1

                    if i == 0:
                        # trajectory.append(np.array(self.env.unwrapped.agent.position, dtype=np.float64))
                        trajectory.append(np.asarray(next_observation[0:2]).copy()) # add agent x,y position
                    
                    observation = next_observation

                    if done:
                        break

                if ep_len >= self.MAX_EPISODE_LEN and not terminated:
                    truncated = True

                next_state = torch.as_tensor(observation, dtype=torch.float32).detach().clone()
                
                self.buffer.states.append(state.squeeze(0).detach().clone())
                self.buffer.actions.append(action.squeeze(0).detach().clone())
                self.buffer.logprobs.append(log_prob.squeeze(0).detach().clone())
                self.buffer.rewards.append(macro_reward)
                self.buffer.state_values.append(state_value.squeeze(0).detach().clone())
                self.buffer.next_states.append(next_state)
                self.buffer.durations.append(duration)
                self.buffer.terminateds.append(bool(terminated))
                self.buffer.truncateds.append(bool(truncated))

                if terminated or truncated:
                    break

            if i == 0:
                # end_pose = observation[2:]
                end_pose = self.env.render()

            total_rewards.append(ep_reward)
            episode_lengths.append(ep_len)

        return (self.buffer, np.mean(total_rewards), np.mean(episode_lengths),
            start_pose, end_pose, trajectory)

    def _capture_rgb_frame(self):
        frame = self.env.render()
        print(f'frame type: {type(frame)}')
        frame = np.asarray(frame)
        if frame.ndim != 3 or frame.shape[2] < 3:
            raise ValueError(f"Expected an RGB image, got shape {frame.shape}")
        return frame[:, :, :3].copy()

    def update_policy_clip(self):

        states = torch.stack(self.buffer.states).detach()
        actions = torch.stack(self.buffer.actions).detach()
        old_logprobs = torch.stack(self.buffer.logprobs).detach()
        values = torch.stack(self.buffer.state_values).detach().float()
        rewards = torch.tensor(self.buffer.rewards, dtype=torch.float32)
        next_states = torch.stack(self.buffer.next_states).detach()
        with torch.no_grad():
            next_values = self.policy.critic(next_states).squeeze(-1)
        durations = torch.tensor(self.buffer.durations, dtype=torch.float32)
        terminateds = torch.tensor(self.buffer.terminateds, dtype=torch.bool)
        episode_ends = terminateds | torch.tensor(self.buffer.truncateds, dtype=torch.bool)

        # calculate gae - generalized advantage estimate
        advantages = torch.zeros_like(rewards)
        gae = 0.0
        for t in reversed(range(len(rewards))):
            bootstrap_mask = 1.0 - terminateds[t].float()
            trace_mask = 1.0 - episode_ends[t].float()
            delta = (
                rewards[t]
                + self.discount_gamma ** durations[t] * next_values[t] * bootstrap_mask
                - values[t]
            )
            gae = delta + (
                (self.discount_gamma * self.gae_lambda) ** durations[t]
                * trace_mask
                * gae
            )
            advantages[t] = gae

        returns = advantages + values
        advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)

        # update policy with gradient ascent
        for _ in range(self.update_epochs):
            logprobs, state_values, entropy = self.policy.evaluate(states, actions)
            ratios = torch.exp(logprobs - old_logprobs)
            surrogate_one = ratios * advantages
            surrogate_two = torch.clamp(
                ratios, 1.0 - self.clip, 1.0 + self.clip
            ) * advantages

            policy_loss = -torch.min(surrogate_one, surrogate_two).mean()
            value_loss = self.loss_fn(state_values, returns)
            entropy_loss = -self.entropy_coefficient * entropy.mean()
            loss = policy_loss + 0.5 * value_loss + entropy_loss

            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            self.optimizer.step()

        self.policy_old.load_state_dict(self.policy.state_dict())
        self.buffer.clear()

    def run(self):
        for i in range(self.max_steps):
            start_time = time.perf_counter()
            _, avg_reward, avg_ep_len, start_pose, end_pose, trajectory = self.collect_dataset()
            end_time = time.perf_counter()
            print(f'Step {i:03d}/{self.max_steps} | Avg Reward: {avg_reward:.4f} | Avg Episode Length: {avg_ep_len:.1f} | collect time: {(end_time - start_time):.2f} s')
            # print(f'start pose: {start_pose} | end_pose: {end_pose} | trajectory: {trajectory}')
            # print(f'start pose type: {type(start_pose[0])} | trajectory type: {type(trajectory[0])} {type(trajectory[0][0])}')
            # observations are numpy float64
            rewards = self.buffer.rewards.copy()
            self.save_rollout_as_jpg(start_pose, end_pose, trajectory, i, rewards)

            start_time = time.perf_counter()
            self.update_policy_clip()
            end_time = time.perf_counter()
            print(f'update policy in {(end_time - start_time):.2f} s')

    def act(self, state):
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        with torch.inference_mode():
            action, log_prob, state_value = self.policy_old.act(state)
        return action, log_prob, state_value

    def save_rollout_as_jpg(self, start_pose, end_pose, trajectory, step, rewards):
        # print("before image conversion", flush=True)
        # start = np.asarray(start_pose)
        # end = np.asarray(end_pose)
        # print("image arrays:", start.shape, start.dtype, end.shape, end.dtype, flush=True)

        # print("before overlay", flush=True)
        # overlay = np.clip(0.5 * start + 0.5 * end, 0, 255).astype(np.uint8)
        # print("after overlay", flush=True)

        # print("before trajectory conversion", flush=True)
        # trajectory = np.asarray(trajectory, dtype=np.float64)
        # print("after trajectory conversion", trajectory.shape, flush=True)

        overlay = np.clip(0.5*start_pose + 0.5 * end_pose, 0, 255).astype(np.uint8)
        trajectory = np.asarray(trajectory, dtype=np.float64)
        rewards = np.asarray(rewards, dtype=np.float64)
        if rewards.size == 0:
            raise ValueError("Cannot plot a reward histogram with no collected rewards")
        # print(f'trajectory shape: {trajectory.shape}')

        fig, (ax, reward_ax) = plt.subplots(
            1, 2, figsize=(13, 6), gridspec_kw={"width_ratios": [1.25, 1]}
        )
        ax.imshow(overlay, extent=(0, 512, 512, 0))
        ax.plot(trajectory[:,0], trajectory[:,1], color="blue", linewidth=2, label="Agent path")
        ax.scatter(*trajectory[0], color="yellow", s=30, label="Agent start")
        ax.scatter(*trajectory[-1], color="red", s=30, label="Agent end")

        ax.set_xlim(0, 512)
        ax.set_ylim(0, 512)
        ax.set_aspect('equal')
        ax.set_title(f'Rollout {step}')

        reward_ax.hist(rewards, bins="auto", color="tab:orange", edgecolor="black")
        reward_ax.set_title(f'Reward Distribution - PPO Iteration {step}')
        reward_ax.set_xlabel('Collected reward per PPO action')
        reward_ax.set_ylabel('Count')
        reward_ax.grid(axis="y", alpha=0.3)

        fig.tight_layout()
        fig.savefig(f"rollout_{step:03d}.jpg")
        plt.close(fig)

    def save(self, checkpoint_path):
        torch.save(self.policy_old.state_dict(), checkpoint_path)

In [127]:
model = PPO(env,
            policy_type='mlp',
            state_dim=state_dim,
            action_dim=action_dim,
            lr_actor=lr_actor,
            lr_critic=lr_critic,
            max_ppo_steps=max_ppo_steps,
            max_episode_len=max_episode_len,
            clip=clip,
            discount_gamma=discount_gamma,
            gae_lambda=gae_lambda,
            update_epochs=update_epochs,
            batch_size=batch_size,
            entropy_coefficient=entropy_coefficient
        )

In [128]:
model.run()

dist_reward 0.11911857720144466 | base_reward: 0.1161716430104556 | progress_reward: 0.0 | angle_progress_reward: 0.0
dist_reward 0.09262648472430751 | base_reward: 0.1161716430104556 | progress_reward: 0.0 | angle_progress_reward: 0.0
dist_reward 0.08308014631426418 | base_reward: 0.1161716430104556 | progress_reward: 0.0 | angle_progress_reward: 0.0
Step 000/100 | Avg Reward: 35.1857 | Avg Episode Length: 300.0 | collect time: 8.29 s
update policy in 0.32 s
dist_reward 0.1125099832611948 | base_reward: 0.09642172674422067 | progress_reward: 0.0 | angle_progress_reward: 0.0
dist_reward 0.10762727816614781 | base_reward: 0.09642172674422067 | progress_reward: 0.0 | angle_progress_reward: 0.0
dist_reward 0.08863612872171273 | base_reward: 0.09642172674422067 | progress_reward: 0.0 | angle_progress_reward: 0.0
Step 001/100 | Avg Reward: 41.9773 | Avg Episode Length: 300.0 | collect time: 8.12 s
update policy in 0.32 s
dist_reward 0.13156744395638587 | base_reward: 0.09608276600209195 | p

In [133]:
def save_frames_as_gif(frames, path, fps=10, frame_stride=1, max_size=(512, 512)):
    from PIL import Image

    if not frames:
        raise ValueError("Cannot save a GIF with no frames")
    if fps <= 0:
        raise ValueError("fps must be positive")
    if frame_stride < 1:
        raise ValueError("frame_stride must be at least 1")

    images = []
    for frame in frames[::frame_stride]:
        frame = np.asarray(frame)
        if frame.ndim != 3 or frame.shape[2] < 3:
            raise ValueError(f"Expected an RGB image, got shape {frame.shape}")
        if frame.dtype != np.uint8:
            frame = np.clip(frame, 0, 255).astype(np.uint8)

        image = Image.fromarray(np.ascontiguousarray(frame[:, :, :3]), mode="RGB")
        if max_size is not None:
            image.thumbnail(max_size, Image.Resampling.BILINEAR)
        images.append(image)

    # GIF timing is quantized to 10 ms; preserve elapsed time when frames are skipped.
    duration = max(10, round((1000 * frame_stride / fps) / 10) * 10)
    images[0].save(
        path,
        save_all=True,
        append_images=images[1:],
        duration=duration,
        loop=0,
        optimize=False,
    )

## TODO: Need to enforce direction consistency and/or pick action every x timesteps

In [134]:
### Test policy and save results as gifs

# asdf = model.save('checkpoint.pt')

# record 10 random rollouts
NUM_TEST_ROLLOUTS = 5
for i in range(NUM_TEST_ROLLOUTS):

    observation, info = env.reset()
    frames = []
    episode_steps = 0
    start_time = time.perf_counter()

    while episode_steps < model.MAX_EPISODE_LEN:
        state = torch.as_tensor(observation, dtype=torch.float32).unsqueeze(0)
        # action = model.policy.act(state)
        with torch.inference_mode():
            action, _, _ = model.policy_old.act(state)
            environment_action = model._to_environment_action(action)

        steps_to_run = min(
            model.action_every_n_steps,
            model.MAX_EPISODE_LEN - episode_steps,
        )
        for _ in range(steps_to_run):
            observation, reward, terminated, truncated, info = env.step(environment_action)
            episode_steps += 1
            frames.append(env.render())
            if terminated or truncated:
                break
        if terminated or truncated:
            break
    end_time = time.perf_counter()
    print(f'episode time elapsed: {(end_time-start_time):.3f}')

    save_frames_as_gif(frames, f'rollout_{i}.gif', fps=10, frame_stride=2, max_size=(256, 256))
    end_time2 = time.perf_counter()
    print(f'gif same time elapsed: {(end_time2 - end_time):.3f}')

episode time elapsed: 1.514
episode time elapsed: 2.331
episode time elapsed: 1.629
episode time elapsed: 2.204
episode time elapsed: 1.708
episode time elapsed: 2.790
episode time elapsed: 1.653
episode time elapsed: 2.605
episode time elapsed: 1.532
episode time elapsed: 2.333
